In [1]:
using Pkg; Pkg.activate("../")

  Activating project at `/central/groups/esm/jschmitt/experiments/feature_importance`


In [2]:
import JLD2
import YAML
import ClimaCalibrate as CAL
using ScikitLearn
using LinearAlgebra

import CalibrateEmulateSample as CES 
using CalibrateEmulateSample.Emulators 
using EnsembleKalmanProcesses.DataContainers

gppackage = Emulators.SKLJL()

SKLJL()

In [3]:
ekp_fpath = "/central/scratch/jschmitt/calibrations/exp2/iteration_008/eki_file.jld2"
eki_obj = JLD2.load_object(ekp_fpath)
prior = CAL.get_prior("/central/groups/esm/jschmitt/ClimaAtmos.jl/calibration/experiments/gcm_driven_scm/prior_prognostic_pi_entr_smooth_entr_detr_impl_0M_v1.toml")

ParameterDistribution with 15 entries: 
'mixing_length_tke_surf_flux_coeff' with EnsembleKalmanProcesses.ParameterDistributions.ConstraintType[Bounds: (0.0, ∞)] over distribution EnsembleKalmanProcesses.ParameterDistributions.Parameterized(Distributions.Normal{Float64}(μ=0.9870405130110049, σ=0.47238072707743883)) 
'max_area_limiter_scale' with EnsembleKalmanProcesses.ParameterDistributions.ConstraintType[Bounds: (0.0, ∞)] over distribution EnsembleKalmanProcesses.ParameterDistributions.Parameterized(Distributions.Normal{Float64}(μ=-4.624780542564731, σ=0.19804220043536505)) 
'mixing_length_diss_coeff' with EnsembleKalmanProcesses.ParameterDistributions.ConstraintType[Bounds: (0.0, ∞)] over distribution EnsembleKalmanProcesses.ParameterDistributions.Parameterized(Distributions.Normal{Float64}(μ=-1.7050130425375234, σ=0.6178758935380921)) 
'specific_humidity_precipitation_threshold' with EnsembleKalmanProcesses.ParameterDistributions.ConstraintType[Bounds: (0.0, ∞)] over distribution En

In [4]:
input_output_pairs = CES.Utilities.get_training_points(eki_obj, 5)

PairedDataContainer{Float64}(DataContainer{Float64}([1.3417544048118368 1.045342003180718 … 1.1427075042011816 1.0048236946575786; -4.489305521367986 -4.478345029238519 … -4.472522051258442 -4.520923767746996; … ; -1.4087638562634435 -1.2758065936647622 … -1.3444793357820064 -1.2150246017439885; -2.3911734843925956 -2.6561266944901574 … -1.0152195324965363 -0.9231952765138505]), DataContainer{Float64}([-0.7973719828160908 -0.7965827523742628 … -0.7973539547095516 -0.7976844699961039; -0.797072659804897 -0.7961132676356127 … -0.7970688159008599 -0.7974351114193305; … ; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265]))

In [5]:
unconstrained_inputs = CES.Utilities.get_inputs(input_output_pairs)
inputs = Emulators.transform_unconstrained_to_constrained(prior, unconstrained_inputs)
size(inputs)

(21, 500)

In [9]:
get_outputs(input_output_pairs)[1:90, :]

90×500 Matrix{Float64}:
 -0.797372  -0.796583  -0.798309  NaN  …  -0.798978  -0.797354  -0.797684
 -0.797073  -0.796113  -0.79809   NaN     -0.798947  -0.797069  -0.797435
 -0.796203  -0.795006  -0.797152  NaN     -0.798297  -0.796221  -0.796551
 -0.795194  -0.793857  -0.796082  NaN     -0.797468  -0.795292  -0.795549
 -0.793914  -0.792663  -0.794816  NaN     -0.796381  -0.794149  -0.794319
 -0.791809  -0.790979  -0.792849  NaN  …  -0.794769  -0.792338  -0.792453
 -0.78728   -0.787522  -0.788724  NaN     -0.79173   -0.788401  -0.788547
 -0.768145  -0.7732    -0.769055  NaN     -0.780211  -0.767035  -0.774846
 -0.650316  -0.669543  -0.643604  NaN     -0.687933  -0.631722  -0.671074
 -0.408703  -0.427412  -0.402322  NaN     -0.439185  -0.385472  -0.41797
  ⋮                                    ⋱                        
 -0.765171  -0.765171  -0.765171  NaN     -0.765171  -0.765171  -0.765171
 -0.765171  -0.765171  -0.765171  NaN     -0.765171  -0.765171  -0.765171
 -0.765171  -0.765171  -

In [7]:

"""
    clean_iopairs_nans(iopairs::PairedDataContainer)

Filters out pairs of inputs and outputs where the output contains `NaN` values.

# Arguments
- `iopairs`: A `PairedDataContainer` with `inputs` and `outputs` data matrices.

# Returns
- A new `PairedDataContainer` with the `NaN`-containing samples removed.
"""
function clean_iopairs_nans(iopairs::PairedDataContainer)
    inputs = get_inputs(iopairs)
    outputs = get_outputs(iopairs)
    
    n_samples_in = size(inputs, 2)
    
    # Identify columns in the output matrix that contain any NaNs.
    # `any` over `dims=1` checks each column. `vec` converts the resulting row matrix to a vector.
    nan_output_cols = vec(any(isnan.(outputs), dims=1))
    
    # Indices of columns to keep (where there are no NaNs)
    kept_indices = findall(.!nan_output_cols)
    
    # Filter inputs and outputs to retain only the "good" columns
    cleaned_inputs = inputs[:, kept_indices]
    cleaned_outputs = outputs[:, kept_indices]
    
    n_removed = n_samples_in - length(kept_indices)
    println("Removed $(n_removed) samples with NaNs out of $(n_samples_in) total.")
    
    return PairedDataContainer(cleaned_inputs, cleaned_outputs)
end

cleaned_iopairs = clean_iopairs_nans(input_output_pairs)


Removed 23 samples with NaNs out of 500 total.


PairedDataContainer{Float64}(DataContainer{Float64}([1.3417544048118368 1.045342003180718 … 1.1427075042011816 1.0048236946575786; -4.489305521367986 -4.478345029238519 … -4.472522051258442 -4.520923767746996; … ; -1.4087638562634435 -1.2758065936647622 … -1.3444793357820064 -1.2150246017439885; -2.3911734843925956 -2.6561266944901574 … -1.0152195324965363 -0.9231952765138505]), DataContainer{Float64}([-0.7973719828160908 -0.7965827523742628 … -0.7973539547095516 -0.7976844699961039; -0.797072659804897 -0.7961132676356127 … -0.7970688159008599 -0.7974351114193305; … ; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265; -0.7651706700379265 -0.7651706700379265 … -0.7651706700379265 -0.7651706700379265]))

In [9]:
get_inputs(cleaned_iopairs)

21×477 Matrix{Float64}:
   1.34175     1.04534     1.04787   …    1.15924     1.14271     1.00482
  -4.48931    -4.47835    -4.45452       -4.35962    -4.47252    -4.52092
  -2.16415    -2.13671    -2.12949       -2.14909    -2.08865    -2.08868
 -12.3258    -12.0488    -12.3345       -12.1286    -12.1901    -12.1805
   7.21473     7.17779     7.29143        7.32566     7.33451     7.32793
   8.34584     3.67696     4.2937    …    1.31249     1.08827     0.409488
  -5.09937    -7.89258    -0.409346      -1.55719    -3.00333    -3.76485
  -9.64345   -13.2828    -12.6649        -6.33821    -7.51235    -7.93924
   5.00926    10.104      10.4644         6.50012     7.50779     7.36859
  -1.80565    -3.13406    -6.33632       -3.54264    -3.8258     -3.13827
   ⋮                                 ⋱                ⋮         
  -1.01096    -0.939704   -1.01774       -1.02039    -0.868093   -0.96033
  -4.02151    -4.58277    -4.27628       -4.15514    -3.99916    -3.81139
   8.22606     8.30013 

In [17]:
# Get all diagonal entries from each observation's covariance matrix
diag_entries = vcat([diag(cov) for cov in eki_obj.observation_series.observations[1].covs[:]])

# Create diagonal matrix from these entries
full_cov_matrix = Diagonal(vcat([diag(hcat(obs.covs...)) for obs in eki_obj.observation_series.observations]...))
# full_cov_matrix = Diagonal(diag(eki_obj.observation_series.observations[1].covs[1]))

gauss_proc = Emulators.GaussianProcess(gppackage, noise_learn = true) 

emulator_gp = Emulator(gauss_proc, 
                    cleaned_iopairs;
                    obs_noise_cov = full_cov_matrix,
                    normalize_inputs = true,
                    retained_svd_frac = 0.95)

optimize_hyperparameters!(emulator_gp)

SVD truncated at k: 33/90
Using default squared exponential kernel, learning length scale and variance parameters
Using default squared exponential kernel:PyObject 1**2 * RBF(length_scale=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])
Learning additive white noise


┌ Info: Training kernel 1, 
└ @ CalibrateEmulateSample.Emulators /home/jschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/GaussianProcess.jl:308
/home/jschmitt/.julia/conda/3/x86_64/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/jschmitt/.julia/conda/3/x86_64/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/jschmitt/.julia/conda/3/x86_64/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1

SKlearn, already trained. continuing...


In [ ]:
get_outputs(cleaned_iopairs)
full_cov_matrix = Diagonal(vcat([diag(hcat(obs.covs...)) for obs in eki_obj.observation_series.observations]...))


nugget = 1e-3
overrides = Dict(
    "verbose" => true,
    # "scheduler" => DataMisfitController(terminate_at = 100.0),
    # "cov_sample_multiplier" => 1.0,
    # "n_iteration" => 8,
    # "n_features_opt" => 40,
)
n_features = 100
n_params = 21
kernel_structure = SeparableKernel(LowRankFactor(1, nugget), OneDimFactor())
mlt = ScalarRandomFeatureInterface(
    n_features,
    n_params,
    kernel_structure = kernel_structure,
    optimizer_options = overrides,
)


emulator_rf = Emulator(mlt, 
                    cleaned_iopairs;
                    obs_noise_cov = full_cov_matrix,
                    normalize_inputs = true,
                    retained_svd_frac = 0.95)

optimize_hyperparameters!(emulator_rf)


┌ Info: hyperparameter optimization with EKI configured with Dict{Any, Any}("inflation" => 0.0001, "localization" => EnsembleKalmanProcesses.Localizers.NoLocalization(), "accelerator" => EnsembleKalmanProcesses.NesterovAccelerator{Float64}(Float64[], 1.0), "scheduler" => EnsembleKalmanProcesses.DataMisfitController{Float64, String}(Int64[], 1000.0, "stop"), "cov_correction" => "shrinkage", "verbose" => true, "multithread" => "ensemble", "n_ensemble" => 100, "cov_sample_multiplier" => 10.0, "n_features_opt" => 100, "train_fraction" => 0.8, "n_cross_val_sets" => 2, "n_iteration" => 10)
└ @ CalibrateEmulateSample.Emulators /home/jschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/ScalarRandomFeature.jl:185


SVD truncated at k: 828/900


┌ Info: hyperparameter learning for 828 models using 381 training points, 96 validation points and 100 features
└ @ CalibrateEmulateSample.Emulators /home/jschmitt/.julia/packages/CalibrateEmulateSample/H2455/src/ScalarRandomFeature.jl:379
